# Guía de Trabajo Práctico 3

**Materia**: Aprendizaje Profundo Basado en la Física (optativa del Instituto Balseiro)

**Docente**: José I. Robledo

**Edición**: abril-mayo 2026

---

## Problema 1: PINN en PyTorch para Poisson 1D


Resolver
$$
u''(x) = -\pi^2 \sin(\pi x), \quad x\in[0,1],\quad u(0)=u(1)=0
$$
cuya solución analítica es `u(x)=sin(pi x)`.

1. Crear una red pequeña `x -> u(x)`.
2. Muestrear puntos interiores y de borde.
3. Usar `autograd` para obtener `u_x` y `u_xx`.
4. Construir una loss con residual de PDE y condiciones de borde.
5. Entrenar y comparar contra la solución analítica.

---

## Problema 2: Poisson 2D con DeepXDE


Resolver con `deepxde` un problema de Poisson 2D sobre el cuadrado unitario .

$$
\Delta u(x,y) = 2\pi^2 \sin(\pi x)\sin(\pi y), \quad (x,y)\in[0,1]^2
$$
con condición de borde homogénea y solución analítica
$$
u(x,y) = -\sin(\pi x)\sin(\pi y).
$$

1. Definir la geometría del cuadrado unitario.
2. Declarar la PDE y la condición de borde de Dirichlet homogénea.
3. Armar el objeto `dde.data.PDE`.
4. Entrenar una red `FNN` que estime la solución en el dominio.
5. Evaluar la solución sobre una grilla y compararla con la expresión analítica.

---

## Problema 2.b: Helmholtz 2D con métrica de prueba y ajuste de hiperparámetros (OPCIONAL)

Retomar el problema anterior de Poisson 2D, cuya solución analítica era

$$
u(x,y) = -\sin(\pi x)\sin(\pi y).
$$

Observar que, para esta solución particular, se cumple que

$$
\Delta u(x,y) = -2\pi^2 u(x,y),
$$

de modo que el mismo campo puede reinterpretarse como solución de una ecuación de Helmholtz homogénea:

$$
-\Delta u(x,y) - k_0^2 u(x,y) = 0, \qquad (x,y)\in[0,1]^2,
$$

con condición de borde de Dirichlet homogénea

$$
u(x,y)=0, \qquad (x,y)\in\partial\Omega,
$$

y con

$$
k_0^2 = 2\pi^2.
$$

Resolver este problema con `DeepXDE`, usando como solución de referencia

$$
u(x,y) = -\sin(\pi x)\sin(\pi y).
$$

1. Definir la geometría rectangular y escribir el residual de la ecuación de Helmholtz usando derivadas automáticas de segundo orden.
2. Declarar la condición de borde de Dirichlet homogénea, o bien imponerla mediante una transformación de salida si se desea trabajar con restricciones duras.
3. Construir el objeto `dde.data.PDE` incluyendo la solución exacta para poder evaluar una métrica de prueba razonable durante el entrenamiento.
4. Entrenar una red `FNN` utilizando, por ejemplo, la métrica `"l2 relative error"` para monitorear la calidad de la solución sobre el conjunto de test.
5. Comparar la solución predicha con la solución analítica sobre una grilla bidimensional y reportar el error final.
6. Realizar una optimización de hiperparámetros inspirada en el [demo oficial de `DeepXDE`](https://deepxde.readthedocs.io/en/v1.15.0/demos/pinn_forward/helmholtz.2d.dirichlet.hpo.html) para Helmholtz 2D, explorando al menos tasa de aprendizaje, cantidad de capas, número de neuronas por capa y función de activación.
7. Definir una función objetivo razonable para la optimización, por ejemplo el mínimo error de test o la mejor métrica `l2 relative error` obtenida durante el entrenamiento.
8. Presentar la mejor configuración encontrada y compararla con una configuración base.

---

## Problema 3: Estimar la difusividad térmica: problema inverso


Se considera la ecuación de difusión unidimensional:
$$
u_t = \alpha u_{xx}
$$

donde $u(x,t)$ representa la temperatura y $\alpha > 0$ es la difusividad térmica desconocida. Supongamos que queremos estimar la solución con una PINN, en el dominio $x \in [0,1]$ y $t \in[0,1]$, con condiciones iniciales $  u(x,0) = \sin(\pi x)$, $u(0,t) = 0$ y $u(1,t) = 0$.

Cargar los datos observados de `data/difusion_1D.csv`. Contiene valores de $(x, t, u(x,t)$ para varios pares $(x,t)$ en el dominio mencionado.  

1. Utilizando `dde.icbc.DirichletBC` para las condiciones de contorno, `dde.icbc.IC` para las condiciones iniciales, y `dde.icbc.PointSetBC` para los puntos de colocacion provistos,  entrenar una PINN dependiente del tiempo con `dde.data.TimePDE` que estime el valor del parámetro $\alpha$. TIP: estas clases están pensadas para trabajar con numpy arrays.

2. Entrenar el modelo durante 5000 épocas con el optimizador Adam, y luego utilizar L-BFGS para el refinamiento de la solución.  

---

## Problema 4: Viga estática simplemente apoyada

Considere una viga unidimensional simplemente apoyada, modelada en forma adimensional por la ecuación de Euler-Bernoulli

$$
\hat{w}_{\xi\xi\xi\xi}(\xi) = 1, \qquad \xi \in (0,1)
$$

donde $\hat{w}(\xi)$ representa la deflexión adimensional de la viga.

Las condiciones de borde para una viga simplemente apoyada son:

$$
\hat{w}(0)=0, \qquad \hat{w}(1)=0,
$$

$$
\hat{w}''(0)=0, \qquad \hat{w}''(1)=0.
$$

La solución analítica del problema es

$$
\hat{w}(\xi) = \frac{\xi\,(1 - 2\xi^2 + \xi^3)}{24}.
$$

Resolver este problema con `DeepXDE` usando el backend de PyTorch.

1. Definir la geometría unidimensional sobre el intervalo $[0,1]$.
2. Implementar la PDE usando derivadas automáticas hasta cuarto orden.
3. Imponer las condiciones de borde esenciales $\hat{w}(0)=\hat{w}(1)=0$ con `dde.icbc.DirichletBC`.
4. Imponer las condiciones de borde naturales $\hat{w}''(0)=\hat{w}''(1)=0$ con `dde.icbc.OperatorBC`.
5. Construir el objeto `dde.data.PDE`, entrenar una red `FNN` primero con Adam y luego refinar con L-BFGS.
6. Evaluar la solución aprendida en una grilla uniforme, compararla con la solución analítica y reportar el error relativo $L^2$.
7. Graficar la solución exacta, la aproximación de la PINN y el error absoluto.

---

## Problema 5: Viga dinámica apoyada

Considere ahora la versión dinámica de la viga apoyada, modelada en variables adimensionales por la ecuación

$$
\hat{w}_{\tau\tau}(\xi,\tau) + \hat{w}_{\xi\xi\xi\xi}(\xi,\tau) = 0,
$$

para $(\xi,\tau) \in (0,1) \times (0,1)$, donde $\hat{w}(\xi,\tau)$ representa la deflexión transversal adimensional.

Las condiciones de borde espaciales de una viga simplemente apoyada son

$$
\hat{w}(0,\tau)=0, \qquad \hat{w}(1,\tau)=0,
$$

$$
\hat{w}_{\xi\xi}(0,\tau)=0, \qquad \hat{w}_{\xi\xi}(1,\tau)=0.
$$

Además, se imponen las condiciones iniciales

$$
\hat{w}(\xi,0)=a\sin(\pi \xi), \qquad \hat{w}_{\tau}(\xi,0)=0,
$$

donde $a>0$ es una amplitud dada.

La solución exacta correspondiente al primer modo es

$$
\hat{w}(\xi,\tau)=a\sin(\pi\xi)\cos(\pi^2\tau).
$$

Resolver este problema con `DeepXDE` usando el backend de PyTorch.

1. Definir el dominio espacio-temporal usando `dde.geometry.Interval`, `dde.geometry.TimeDomain` y `dde.geometry.GeometryXTime`.
2. Implementar la PDE dinámica usando derivadas automáticas de segundo orden en tiempo y cuarto orden en espacio.
3. Imponer las condiciones de borde esenciales $\hat{w}=0$ con `dde.icbc.DirichletBC` y las condiciones naturales $\hat{w}_{\xi\xi}=0$ con `dde.icbc.OperatorBC`.
4. Imponer las condiciones iniciales de desplazamiento y velocidad usando `dde.icbc.IC` y `dde.icbc.OperatorBC`.
5. Construir un objeto `dde.data.TimePDE`, entrenar una red `FNN` con Adam, usar pesos apropiados en la loss. OPCIONAL: puede investigar como implementar un callback `dde.callbacks.PDEPointResampler` para remuestrear puntos de colocación cada 200 iteraciones.
6. Refinar la solución con el optimizador `L-BFGS`.
7. Evaluar el error relativo $L^2$ para distintos tiempos sobre una grilla espacial uniforme.
8. Graficar perfiles de la solución PINN para varios tiempos y comparar la solución exacta con la aproximación para un tiempo fijo.
9. OPCIONAL: Generar una animación o GIF que muestre la evolución temporal de la deformación de la viga y la comparación entre solución exacta y PINN.

<!-- TODO (Markdown): comparación entre PyTorch manual, DeepXDE y PINA -->